Memastikan jumlah baris tiap partisipan sama dengan melakukan teknik truncating

In [ ]:
# Tidak perlu di-run, hanya code arsip

# TRUNCATING:
# Menghapus baris tambahan dari partisipan dengan jumlah baris yang lebih banyak hingga mencapai jumlah baris minimum.

# Load the CSV file
df = pd.read_csv('Dataset/Dataset-ts-Pritalia2020.csv')

# Determine the minimum number of rows for each group
min_rows = df.groupby('nama').size().min()

# Function to truncate each group to the minimum number of rows
def truncate_group(group):
    return group.iloc[:min_rows]

# Apply the truncation function to each group
truncated_df = df.groupby('nama').apply(truncate_group).reset_index(drop=True)

# Save the truncated dataframe to a new CSV file
truncated_df.to_csv('Dataset/data-split/truncated_dataset-seqglo.csv', index=False)

print(f"Truncated dataset saved to 'truncated_dataset-seqglo.csv'.")

Men-cek jumlah partisipan

In [ ]:
import pandas as pd

# Baca data
df = pd.read_csv('Dataset/data-split/truncated_dataset-seqglo.csv')

# Hitung partisipan unik
jumlah_partisipan = df['nama'].nunique()
print(f"Jumlah partisipan unik: {jumlah_partisipan}")


kode Python untuk mengecek jumlah baris (record) dalam suatu dataset (CSV):

In [ ]:
import pandas as pd

# Ganti dengan path ke file kamu
df = pd.read_csv('Dataset/Dataset-ts-Pritalia2020.csv')

# Cek jumlah baris
print(f'Jumlah baris: {len(df)}')


# Split Data 

Mencetak nama dan label di dataset asli

In [ ]:
label_dist = df.groupby('nama')['label'].value_counts().unstack().fillna(0)
label_dist.columns = ['label_1', 'label_2']
label_dist['total'] = label_dist['label_1'] + label_dist['label_2']
label_dist = label_dist.sort_values(by='total', ascending=False)

# Menyimpan ke file CSV
label_dist.to_csv('Dataset/data-split/label_distribution.csv', index=True)

print(label_dist)


Membagi nama-nama partisipan di setiap split dengan data training yang imbang antara label 1 dan label 2.

In [ ]:
import pandas as pd
import os

# Baca dataset utama
df = pd.read_csv('Dataset/Dataset-ts-Pritalia2020.csv')

# Baca file pembagian 5 split
split_df = pd.read_csv('Dataset/data-split/data_split_5x.csv')

# Pastikan kolom nama cocok formatnya
df['nama'] = df['nama'].astype(str)
split_df['nama'] = split_df['nama'].astype(str)

# Buat folder output jika belum ada
output_dir = 'Dataset/Split'
os.makedirs(output_dir, exist_ok=True)

# Proses tiap split
for i in range(1, 6):
    split_name = f'split_{i}'
    
    current_split = split_df[split_df['split'] == split_name]
    train_names = current_split[current_split['set'] == 'train']['nama']
    test_names = current_split[current_split['set'] == 'test']['nama']

    train_df = df[df['nama'].isin(train_names)]
    test_df = df[df['nama'].isin(test_names)]

    train_df.to_csv(f'{output_dir}/train_{split_name}.csv', index=False)
    test_df.to_csv(f'{output_dir}/test_{split_name}.csv', index=False)

    print(f'{split_name}: Train = {len(train_df)} rows, Test = {len(test_df)} rows')


# Memastikan data training imbang

In [1]:
import pandas as pd

for i in range(1, 6):
    print(f"🔍 Fold {i}")
    
    train_df = pd.read_csv(f'Dataset/Split/train_split_{i}.csv')
    test_df = pd.read_csv(f'Dataset/Split/test_split_{i}.csv')

    print("Training set:")
    print(train_df['label'].value_counts())
    
    print("Testing set:")
    print(test_df['label'].value_counts())
    
    print("-" * 30)


🔍 Fold 1
Training set:
label
1    102285
2    102228
Name: count, dtype: int64
Testing set:
label
2    25585
1    25575
Name: count, dtype: int64
------------------------------
🔍 Fold 2
Training set:
label
1    102291
2    102236
Name: count, dtype: int64
Testing set:
label
2    25577
1    25569
Name: count, dtype: int64
------------------------------
🔍 Fold 3
Training set:
label
1    102314
2    102235
Name: count, dtype: int64
Testing set:
label
2    25578
1    25546
Name: count, dtype: int64
------------------------------
🔍 Fold 4
Training set:
label
1    102317
2    102239
Name: count, dtype: int64
Testing set:
label
2    25574
1    25543
Name: count, dtype: int64
------------------------------
🔍 Fold 5
Training set:
label
2    102307
1    102299
Name: count, dtype: int64
Testing set:
label
1    25561
2    25506
Name: count, dtype: int64
------------------------------


Memastikan jumlah baris sama antar partisipan

In [ ]:
# Tidak perlu selalu di-run, cukup sekali untuk mendapatkan info jumlah baris per partisipan

# MEMASTIKAN JUMLAH BARIS SAMA

# Input file datasetnya
df = pd.read_csv('truncated_dataset-seqglo.csv')

# Menghitung jumlah baris per partisipan
jumlah_per_partisipan = df['nama'].value_counts()

# Hasil
print("Jumlah baris per partisipan:")
print(jumlah_per_partisipan)


# Menyimpan hasil ke dalam file Excel
jumlah_per_partisipan.to_excel('jumlah_per_partisipan-seqglo.xlsx', sheet_name='Sheet1')

print("Hasil telah disimpan dalam file jumlah_per_partisipan.xlsx")

# Feature Engineering

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('Dataset/Split/test_split_1.csv')

In [3]:
# Konversi kolom 'gazeX' dan 'gazeY' menjadi tipe data numerik
df['gazeX'] = pd.to_numeric(df['gazeX'], errors='coerce')  # errors='coerce' untuk mengubah nilai yang tidak dapat diubah menjadi NaN
df['gazeY'] = pd.to_numeric(df['gazeY'], errors='coerce')


In [4]:
# Mengekstrak fitur kecepatan dan arah

# Menghitung selisih antara satu baris dengan baris berikutnya
df['delta_gazeX'] = df['gazeX'].diff()
df['delta_gazeY'] = df['gazeY'].diff()

# Menghitung kuadrat total
df['squared_sum'] = df['delta_gazeX']**2 + df['delta_gazeY']**2

# Membagi hasil kuadrat total dengan 0.01667 dan menghitung akar kuadrat
df['kecepatan'] = (df['squared_sum'] / 0.01667)**0.5

# Menghitung nilai arah (direction) menggunakan rumus yang diberikan
df['direction'] = np.arctan2(df['delta_gazeY'], df['delta_gazeX'])

# Mengisi nilai NaN pada baris pertama dengan 0
df['kecepatan'] = df['kecepatan'].fillna(0)
df['direction'] = df['direction'].fillna(0)

# Tampilkan DataFrame dengan fitur kecepatan
print(df)

         nama   time      gazeX     gazeY  label  delta_gazeX  delta_gazeY  \
0      Andika     17   293.4336  141.1452      2          NaN          NaN   
1      Andika     33   293.4336  141.1452      2       0.0000       0.0000   
2      Andika     50   133.2864   45.5328      2    -160.1472     -95.6124   
3      Andika     66   130.2528  197.6076      2      -3.0336     152.0748   
4      Andika     82   126.3744  192.2616      2      -3.8784      -5.3460   
...       ...    ...        ...       ...    ...          ...          ...   
51155    Revi  59914  1686.0100  625.8492      1      -6.1050     -21.8052   
51156    Revi  59931  1687.1620  625.6440      1       1.1520      -0.2052   
51157    Revi  59947  1669.8240  619.6716      1     -17.3380      -5.9724   
51158    Revi  59964  1700.8700  630.9576      1      31.0460      11.2860   
51159    Revi  59980  1703.5010  628.3332      1       2.6310      -2.6244   

        squared_sum    kecepatan  direction  
0               N

In [5]:
# Mengekstrak fitur acceleration

# Menghitung selisih antara satu baris dengan baris berikutnya
df['delta_gazeX'] = df['gazeX'].diff()
df['delta_gazeY'] = df['gazeY'].diff()

# Menghitung kecepatan antara satu baris dengan baris berikutnya
df['kecepatanX'] = df['delta_gazeX']/(0.01667)
df['kecepatanY'] = df['delta_gazeY']/(0.01667)

# Hitung perubahan kecepatan (delta v)
df['delta_vX'] = df['kecepatanX'].diff()
df['delta_vY'] = df['kecepatanY'].diff()

# Menghitung percepatan
dt = 0.01667  # durasi waktu antar sampel
df['acceleration'] = np.sqrt((df['delta_vX'] / dt) ** 2 + (df['delta_vY'] / dt) ** 2)

# Mengisi nilai NaN pada baris pertama dengan 0
df['acceleration'] = df['acceleration'].fillna(0)

# Tampilkan DataFrame dengan fitur kecepatan, arah, dan percepatan
print(df)


         nama   time      gazeX     gazeY  label  delta_gazeX  delta_gazeY  \
0      Andika     17   293.4336  141.1452      2          NaN          NaN   
1      Andika     33   293.4336  141.1452      2       0.0000       0.0000   
2      Andika     50   133.2864   45.5328      2    -160.1472     -95.6124   
3      Andika     66   130.2528  197.6076      2      -3.0336     152.0748   
4      Andika     82   126.3744  192.2616      2      -3.8784      -5.3460   
...       ...    ...        ...       ...    ...          ...          ...   
51155    Revi  59914  1686.0100  625.8492      1      -6.1050     -21.8052   
51156    Revi  59931  1687.1620  625.6440      1       1.1520      -0.2052   
51157    Revi  59947  1669.8240  619.6716      1     -17.3380      -5.9724   
51158    Revi  59964  1700.8700  630.9576      1      31.0460      11.2860   
51159    Revi  59980  1703.5010  628.3332      1       2.6310      -2.6244   

        squared_sum    kecepatan  direction   kecepatanX   kece

In [6]:
# sementara fitur dengan akurasi terbaik
# Cumulative Distance

# Menghitung selisih antara satu baris dengan baris berikutnya
df['delta_gazeX'] = df['gazeX'].diff()
df['delta_gazeY'] = df['gazeY'].diff()

# Menghitung kuadrat selisih
df['squared_diff'] = df['delta_gazeX']**2 + df['delta_gazeY']**2

# Menghitung perpindahan (displacement) menggunakan rumus yang diberikan
df['cumulative-distance'] = np.sqrt(df['squared_diff'].cumsum()) # cumsum akan memjumlahkan baris k+(k+1)

# Mengisi nilai NaN pada baris pertama dengan 0
df['cumulative-distance'] = df['cumulative-distance'].fillna(0)

# Tampilkan DataFrame dengan fitur kecepatan, arah, percepatan, dan displacement
print(df)


         nama   time      gazeX     gazeY  label  delta_gazeX  delta_gazeY  \
0      Andika     17   293.4336  141.1452      2          NaN          NaN   
1      Andika     33   293.4336  141.1452      2       0.0000       0.0000   
2      Andika     50   133.2864   45.5328      2    -160.1472     -95.6124   
3      Andika     66   130.2528  197.6076      2      -3.0336     152.0748   
4      Andika     82   126.3744  192.2616      2      -3.8784      -5.3460   
...       ...    ...        ...       ...    ...          ...          ...   
51155    Revi  59914  1686.0100  625.8492      1      -6.1050     -21.8052   
51156    Revi  59931  1687.1620  625.6440      1       1.1520      -0.2052   
51157    Revi  59947  1669.8240  619.6716      1     -17.3380      -5.9724   
51158    Revi  59964  1700.8700  630.9576      1      31.0460      11.2860   
51159    Revi  59980  1703.5010  628.3332      1       2.6310      -2.6244   

        squared_sum    kecepatan  direction   kecepatanX   kece

In [7]:
# Displacement

# Hitung displacement antar dua titik berurutan
df['deltaX'] = df['gazeX'].diff().shift(-1)
df['deltaY'] = df['gazeY'].diff().shift(-1)
df['displacement'] = np.sqrt(df['deltaX']**2 + df['deltaY']**2)

# Drop kolom tambahan jika tidak ingin disimpan
df = df.drop(columns=['deltaX', 'deltaY'])

# Mengisi nilai NaN pada baris pertama dengan 0
df['displacement'] = df['displacement'].fillna(0)

# Tampilkan DataFrame dengan fitur kecepatan, arah, percepatan, dan displacement
print(df)


         nama   time      gazeX     gazeY  label  delta_gazeX  delta_gazeY  \
0      Andika     17   293.4336  141.1452      2          NaN          NaN   
1      Andika     33   293.4336  141.1452      2       0.0000       0.0000   
2      Andika     50   133.2864   45.5328      2    -160.1472     -95.6124   
3      Andika     66   130.2528  197.6076      2      -3.0336     152.0748   
4      Andika     82   126.3744  192.2616      2      -3.8784      -5.3460   
...       ...    ...        ...       ...    ...          ...          ...   
51155    Revi  59914  1686.0100  625.8492      1      -6.1050     -21.8052   
51156    Revi  59931  1687.1620  625.6440      1       1.1520      -0.2052   
51157    Revi  59947  1669.8240  619.6716      1     -17.3380      -5.9724   
51158    Revi  59964  1700.8700  630.9576      1      31.0460      11.2860   
51159    Revi  59980  1703.5010  628.3332      1       2.6310      -2.6244   

        squared_sum    kecepatan  direction   kecepatanX   kece

In [8]:
# Standar Deviasi  

import pandas as pd
import numpy as np

# Panjang window
window = 2

# Inisialisasi array hasil
stddev_values = np.zeros(len(df))

# Sliding window non-overlapping (melompat per 2 baris)
for i in range(0, len(df) - window + 1, window):
    window_x = df['gazeX'].iloc[i:i+window]
    window_y = df['gazeY'].iloc[i:i+window]
    
    std_x = np.std(window_x, ddof=0)  # populasi
    std_y = np.std(window_y, ddof=0)  # populasi
    
    std_combined = np.sqrt(std_x**2 + std_y**2)
    
    # Simpan hasil ke dalam window
    stddev_values[i:i+window] = std_combined

# Tambahkan ke DataFrame
df['stddev_2pop'] = stddev_values

# Tampilkan hasil
print(df)


         nama   time      gazeX     gazeY  label  delta_gazeX  delta_gazeY  \
0      Andika     17   293.4336  141.1452      2          NaN          NaN   
1      Andika     33   293.4336  141.1452      2       0.0000       0.0000   
2      Andika     50   133.2864   45.5328      2    -160.1472     -95.6124   
3      Andika     66   130.2528  197.6076      2      -3.0336     152.0748   
4      Andika     82   126.3744  192.2616      2      -3.8784      -5.3460   
...       ...    ...        ...       ...    ...          ...          ...   
51155    Revi  59914  1686.0100  625.8492      1      -6.1050     -21.8052   
51156    Revi  59931  1687.1620  625.6440      1       1.1520      -0.2052   
51157    Revi  59947  1669.8240  619.6716      1     -17.3380      -5.9724   
51158    Revi  59964  1700.8700  630.9576      1      31.0460      11.2860   
51159    Revi  59980  1703.5010  628.3332      1       2.6310      -2.6244   

        squared_sum    kecepatan  direction   kecepatanX   kece

In [9]:
# Linearity Index (LI)

# Asumsikan df sudah punya kolom 'displacement' dan 'cumulative-distance'
df['linearity_index'] = df['displacement'] / df['cumulative-distance']

df['linearity_index'] = df.apply(
    lambda row: row['displacement'] / row['cumulative-distance'] if row['cumulative-distance'] != 0 else 0,
    axis=1
)



In [10]:
# Multivariat

# menguji fitur kecepatan, arah, percepatan, standard deviation, cumulative-distance dan displacement

X0 = df.drop(['nama','time', 'label', 'delta_gazeX','delta_gazeY','squared_diff','squared_sum', 'kecepatanX', 'kecepatanY', 'delta_vX', 'delta_vY'], axis=1)
y0 = df['label']


In [11]:
X0

,gazeX,gazeY,kecepatan,direction,acceleration,cumulative-distance,displacement,stddev_2pop,linearity_index
0,293.4336,141.1452,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
1,293.4336,141.1452,0.000000,0.000000,0.000000e+00,0.000000,186.517711,0.000000,0.000000
2,133.2864,45.5328,1444.615526,-2.603361,6.711953e+05,186.517711,152.105054,76.052527,0.815499
3,130.2528,197.6076,1178.082881,1.590742,1.055511e+06,240.675724,6.604673,76.052527,0.027442
4,126.3744,192.2616,51.154460,-2.198417,5.664964e+05,240.766331,756.186379,378.093190,3.140748
...,...,...,...,...,...,...,...,...,...
51155,1686.0100,625.8492,175.379915,-1.843786,5.244545e+05,31791.205734,1.170133,11.321857,0.000037
51156,1687.1620,625.6440,9.062904,-0.176276,8.199856e+04,31791.205755,18.337824,9.168912,0.000577
51157,1669.8240,619.6716,142.029975,-2.809854,6.969889e+04,31791.211044,33.033739,9.168912,0.001039
51158,1700.8700,630.9576,255.852658,0.348673,1.848576e+05,31791.228207,3.716132,1.858066,0.000117


In [12]:
y0

0        2
1        2
2        2
3        2
4        2
        ..
51155    1
51156    1
51157    1
51158    1
51159    1
Name: label, Length: 51160, dtype: int64

In [13]:
import pandas as pd

# Simpan X0 sebagai file CSV
X0.to_csv('Dataset/data-split/feature-engineering/X0-test-split1-seqglo-truncate.csv', index=False)

# Simpan y0 sebagai file CSV
y0.to_csv('Dataset/data-split/feature-engineering/y0-test-split1-seqglo-truncate.csv', index=False)
